In [17]:
# checking system info
import sys

# for running gradient descent
import numpy as np
import torch
import cvxpy as cp
from cvxpylayers.torch import CvxpyLayer

# for bringing the data in from R
import pandas as pd

import learnQ as lq


In [18]:
print(sys.executable)

/Users/jseid1/venv311/bin/python


## Synthetic Data Experiment (Using my DGP)

In [20]:
# read in the data
data = pd.read_csv('outcomes.csv')
metadata = pd.read_csv("metadata.csv")

# convert to numpy arrays
metadata = metadata.to_numpy()
data = data.to_numpy()

# unpack the metadata
n_units, n_timepoints, n_outcomes = metadata[:,1]

# reshape the data
Y = data.reshape(n_timepoints, n_outcomes, n_units)  # The data now has shape - (n_timepoints, n_outcomes, n_units)


test = Y[len(Y)-1] # take the last time point - Has shape (outcomes x units) (K,N)
train = Y[0:len(Y)-1] # take all covariate matrices up until the last time point - for each time point has shape (outcomes x units) (T-1, K, N)

verbose=False
if (verbose):
    print("Test data: \n\n", test)
    print("\nTrain data: \n\n", train)
    # print(train.shape) #  It is (timepoints-1, outcomes, units)

# split the data into the target vectors and the covariate matrices
# goal here is to use final outcomes to predict final target
train_target_vectors = [torch.from_numpy(train[i][0:train[i].shape[0],0:1]).squeeze() for i in range(len(train))]
train_covariate_matrices = [torch.from_numpy(train[i][0:train[i].shape[0],1:train[i].shape[1]]) for i in range(len(train))]

verbose=False
if verbose == True:
    for i in range(len(train)):
        print("\nTarget vector", i, ":\n", train_target_vectors[i])
        print("\nCovariate matrix", i, ":\n", train_covariate_matrices[i])


# The treated unit at the final time point
test_target_vector = torch.from_numpy(test[0:test.shape[0],0:1]).squeeze()
test_covariate_matrix = torch.from_numpy(test[0:test.shape[0],1:test.shape[1]])

# run the learn Q function:
# NB: embedding into a much lower dimension also works! (try 2)
# embedding dimension doesn't make that much of a difference interestingly
Q_final, w_final = lq.learnQ(train_target_vectors, train_covariate_matrices, 5, 2000, 1, 10, False)

torch.set_printoptions(sci_mode=False, precision=2)
print("\n train original covariates:\n", train_covariate_matrices)

print("\n test [Original] covariates: \n", test_covariate_matrix)
print("\n test [Synthetic] covariates: \n",test_covariate_matrix@Q_final)
print("\nFinal weights: \n", w_final.round(2))
print("\nComputed: \n", test_covariate_matrix@Q_final@w_final)
print("\nGoal: \n", test_target_vector)
print("Final Q: \n", Q_final.round(2))

if verbose == True:
    for i in range(len(train_target_vectors)): 
        print("Final Q: \n", Q_final)
        print("\nTraining Sample ", i)
        print("Goal: \n", train_target_vectors[i])
        print("Computed \n", train_covariate_matrices[i] @ Q_final @ w_final)
        print("\n")



Step    0 | Loss: 465.66724980
Step    0 | Loss: 465.66724980 | w: [ 0.717  0.256 -0.     0.027 -0.   ]
Grad norm: 799.83580399
Step  200 | Loss: 89.15601712
Step  200 | Loss: 89.15601712 | w: [ 0.724  0.276 -0.    -0.    -0.   ]
Grad norm: 11.24974309
Step  400 | Loss: 80.99172213
Step  400 | Loss: 80.99172213 | w: [ 0.632  0.368 -0.    -0.    -0.   ]
Grad norm: 5.15061174
Step  600 | Loss: 79.21670527
Step  600 | Loss: 79.21670527 | w: [ 0.563  0.437 -0.    -0.    -0.   ]
Grad norm: 3.37130342
Step  800 | Loss: 78.60544034
Step  800 | Loss: 78.60544034 | w: [ 0.534  0.466  0.    -0.     0.   ]
Grad norm: 2.70727349
Step 1000 | Loss: 78.19583548
Step 1000 | Loss: 78.19583548 | w: [ 0.518  0.483 -0.    -0.    -0.   ]
Grad norm: 2.18391862
Step 1200 | Loss: 78.03711876
Step 1200 | Loss: 78.03711876 | w: [0.511 0.489 0.    0.    0.   ]
Grad norm: 1.94883185
Step 1400 | Loss: 77.89825788
Step 1400 | Loss: 77.89825788 | w: [ 0.503  0.497 -0.    -0.    -0.   ]
Grad norm: 1.72655691
Step 160